<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/predict_traditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


NIEKROCZACY FORECAST

In [9]:
%%writefile /content/drive/MyDrive/ml_project/predict_traditional.py
# -*- coding: utf-8 -*-
import os
import pickle
import numpy as np
import pandas as pd

target_dir = "/content/drive/MyDrive/ml_project"

model_path = os.path.join(target_dir, "traditional_trained.pkl")
y_train_path = os.path.join(target_dir, "y_train.pkl")
y_pred_path = os.path.join(target_dir, "y_pred_traditional.pkl")


# ---------------------------------------------------------
# predict_traditional
# ---------------------------------------------------------
def predict_traditional():

    if not os.path.exists(model_path):
        raise FileNotFoundError("traditional_trained.pkl not found")

    if not os.path.exists(y_train_path):
        raise FileNotFoundError("y_train.pkl not found")

    # load trained model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # load y_train (needed for naive and moving average)
    y_train = pd.read_pickle(y_train_path)
    y_series = pd.Series(y_train).dropna()

    # h = forecast horizon (length of y_test)
    y_test_path = os.path.join(target_dir, "y_test.pkl")
    if not os.path.exists(y_test_path):
        raise FileNotFoundError("y_test.pkl not found")
    y_test = pd.read_pickle(y_test_path)
    h = len(y_test)

    # ---------------------------------------------------------
    # naive PONIZE FORECAST JEDEN TEN SAM NA CALY OKRES
    # ---------------------------------------------------------
    if isinstance(model, dict) and model.get("type") == "naive":
        last_value = y_series.iloc[-1]
        y_pred = np.array([last_value] * h)  # UWAGA PODMIENIAM

    # Naive z kroczącą aktualizacją na podstawie y_test  JEST OKE!!!TO prawidlowy krczacy algorytm
    # if isinstance(model, dict) and model.get("type") == "naive":
    #     y_pred = []
    #     current_value = y_series.iloc[-1]  # Ostatnia wartość z treningu jako start
    #
    #     for i in range(h):
    #         y_pred.append(current_value)
    #
    #         # Aktualizujemy bieżącą wartość na podstawie rzeczywistej danej testowej, jeśli jest dostępna
    #         if i < len(y_test):
    #             current_value = y_test.iloc[i]
    #         else:
    #             # Jeśli skończą się dane testowe, pozostajemy przy ostatniej prognozowanej wartości
    #             current_value = current_value
    #
    #     y_pred = np.array(y_pred)

    # ---------------------------------------------------------
    # moving average
    # ---------------------------------------------------------
    elif isinstance(model, dict) and model.get("type") == "moving_average":
        w = model["window"]
        ma_value = y_series.iloc[-w:].mean()
        y_pred = np.array([ma_value] * h)  # UWAGA TEZ ZMIANA NA KROCZACAMETODE

    # elif isinstance(model, dict) and model.get("type") == "moving_average":
    #     w = model["window"]
    #
    #     # start: historia z treningu
    #     history = list(y_series.values)
    #     y_pred = []
    #
    #     for i in range(h):
    #         # 1. prognoza na krok i (tylko z przeszłości)
    #         ma = np.mean(history[-w:])
    #         y_pred.append(ma)
    #
    #         # 2. aktualizacja historii PRAWDZIWĄ wartością z testu
    #         history.append(y_test.iloc[i])
    #
    #     y_pred = np.array(y_pred)

    # ---------------------------------------------------------
    # ses
    # ---------------------------------------------------------
    elif str(type(model)).endswith("SimpleExpSmoothingResults'>"):
        y_pred = model.forecast(h)
        y_pred = np.array(y_pred)

    # elif str(type(model)).endswith("SimpleExpSmoothingResults'>"):
    #     alpha = model.params['smoothing_level']
    #
    #     # start: ostatni level z treningu
    #     if len(model.fittedvalues) > 0:
    #         level = float(model.fittedvalues.iloc[-1])
    #     else:
    #         level = float(y_series.iloc[-1])
    #
    #     y_pred = []
    #
    #     for i in range(h):
    #         # 1. prognoza na krok i
    #         forecast = level
    #         y_pred.append(forecast)
    #
    #         # 2. aktualizacja level PRAWDZIWĄ wartością z testu
    #         y_real = float(y_test.iloc[i])
    #         level = alpha * y_real + (1 - alpha) * level
    #
    #     y_pred = np.array(y_pred)

    # ---------------------------------------------------------
    # arima
    # ---------------------------------------------------------
    elif hasattr(model, "forecast"):
        y_pred = model.forecast(steps=h)
        y_pred = np.array(y_pred)

    else:
        raise ValueError("unknown traditional model type")

    # save predictions
    with open(y_pred_path, "wb") as f:
        pickle.dump(y_pred, f)

    print("saved:", y_pred_path)
    print("traditional predictions generated")

    return y_pred

    # elif hasattr(model, "forecast"):
    #     y_pred = []
    #
    #     # start: model wytrenowany na y_train
    #     current_model = model
    #
    #     for i in range(h):
    #         # 1. prognoza na krok i
    #         forecast = current_model.forecast(steps=1)[0]
    #         y_pred.append(forecast)
    #
    #         # 2. aktualizacja modelem PRAWDZIWĄ wartością z testu
    #         y_real = float(y_test.iloc[i])
    #         current_model = current_model.append([y_real], refit=False)
    #
    #     y_pred = np.array(y_pred)


Overwriting /content/drive/MyDrive/ml_project/predict_traditional.py


KROCZACY FORECAST

In [10]:
%%writefile /content/drive/MyDrive/ml_project/predict_traditional.py
# -*- coding: utf-8 -*-
import os
import pickle
import numpy as np
import pandas as pd

target_dir = "/content/drive/MyDrive/ml_project"

model_path = os.path.join(target_dir, "traditional_trained.pkl")
y_train_path = os.path.join(target_dir, "y_train.pkl")
y_pred_path = os.path.join(target_dir, "y_pred_traditional.pkl")


# ---------------------------------------------------------
# predict_traditional
# ---------------------------------------------------------
def predict_traditional():

    if not os.path.exists(model_path):
        raise FileNotFoundError("traditional_trained.pkl not found")

    if not os.path.exists(y_train_path):
        raise FileNotFoundError("y_train.pkl not found")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    y_train = pd.read_pickle(y_train_path)
    y_series = pd.Series(y_train).dropna()

    y_test_path = os.path.join(target_dir, "y_test.pkl")
    if not os.path.exists(y_test_path):
        raise FileNotFoundError("y_test.pkl not found")
    y_test = pd.read_pickle(y_test_path)
    h = len(y_test)

    # ---------------- naive ----------------
    if isinstance(model, dict) and model.get("type") == "naive":
        y_pred = []
        current_value = y_series.iloc[-1]

        for i in range(h):
            y_pred.append(current_value)
            current_value = y_test.iloc[i]

        y_pred = np.array(y_pred)

    # ---------------- moving average ----------------
    elif isinstance(model, dict) and model.get("type") == "moving_average":
        w = model["window"]
        history = list(y_series.values)
        y_pred = []

        for i in range(h):
            ma = np.mean(history[-w:])
            y_pred.append(ma)
            history.append(y_test.iloc[i])

        y_pred = np.array(y_pred)

    # ---------------- ses ----------------


Overwriting /content/drive/MyDrive/ml_project/predict_traditional.py
